# Assignment 04 — Bài toán 3: Phân loại Đánh giá khách hàng bằng CNN 1D trên văn bản

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **CNN 1D cho VĂN BẢN** bằng NumPy from scratch, gồm cả **tầng nhúng
   (Embedding)** với lan truyền ngược thưa — thành phần mà hai bài trước không có.
2. Dựng hai bản tương đương bằng **PyTorch** và **TensorFlow/Keras**.
3. Xử lý bài toán **đa lớp mất cân bằng nặng** (lớp hiếm nhất chỉ chiếm 0,5%)
   bằng trọng số lớp, và đánh giá bằng **Macro-F1 / Balanced Accuracy** thay vì
   Accuracy thuần.
4. **Đây là bài toán mà tích chập thực sự có lý do tồn tại.** Khác hai bài dữ
   liệu bảng, thứ tự token trong câu **có** ý nghĩa thật: `[not, good]` khác
   `[very, good]`. Ta chứng minh điều này bằng thí nghiệm **xáo trộn thứ tự từ**.
5. Xuất `model_cnn.json` và gọi từ frontend React để chạy serverless trên Vercel.

**Bài toán:** Multiclass Classification — dự đoán `Department Name` (6 lớp) chỉ
từ **nội dung đánh giá bằng văn bản**.

> **Rò rỉ nhãn.** Cột `Class Name` xác định `Department Name` một–một
> (Jeans → Bottoms), nên **bắt buộc phải loại bỏ**. Mô hình ở đây dùng *duy nhất*
> trường văn bản, không dùng bất kỳ đặc trưng bảng nào.

In [1]:
# ============================================================================
# KHỐI 1 — Nạp thư viện, cố định seed và cấu hình hình vẽ
# ============================================================================
import json
import os
import pathlib
import re
import time
from collections import Counter

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
)

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("deep")

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc :", ROOT)
print("Thư mục hình vẽ  :", FIG)

Thư mục làm việc : C:\Users\admin\Downloads\bt-thay quế\tuan 2\customer-behavior-predict
Thư mục hình vẽ  : C:\Users\admin\Downloads\bt-thay quế\tuan 2\customer-behavior-predict\figures


## 1. Nạp dữ liệu và khảo sát

In [2]:
# ============================================================================
# KHỐI 2 — Nạp dữ liệu, bỏ dòng thiếu văn bản hoặc thiếu nhãn
# ============================================================================
df = pd.read_csv(ROOT / "ml" / "data" / "ecommerce_raw.csv")
print("Kích thước gốc:", df.shape)

df = df.dropna(subset=["Review Text", "Department Name"]).reset_index(drop=True)
print("Sau khi bỏ dòng thiếu văn bản/nhãn:", df.shape)

CLASSES = sorted(df["Department Name"].unique())
N_CLASS = len(CLASSES)
CLASS_VI = {
    "Bottoms": "Quần / Chân váy",
    "Dresses": "Đầm",
    "Intimate": "Đồ lót & mặc nhà",
    "Jackets": "Áo khoác",
    "Tops": "Áo",
    "Trend": "Hàng xu hướng",
}
print(f"\n{N_CLASS} lớp: {CLASSES}\n")
dist = df["Department Name"].value_counts()
print(dist.to_string())
print(f"\nMất cân bằng: lớp lớn nhất / nhỏ nhất = {dist.max() / dist.min():.1f} lần")
print(f"Lớp hiếm nhất '{dist.idxmin()}' chỉ chiếm {100*dist.min()/len(df):.2f}% dữ liệu")

Kích thước gốc: (23486, 11)
Sau khi bỏ dòng thiếu văn bản/nhãn: (22628, 11)

6 lớp: ['Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend']

Department Name
Tops        10048
Dresses      6145
Bottoms      3662
Intimate     1653
Jackets      1002
Trend         118

Mất cân bằng: lớp lớn nhất / nhỏ nhất = 85.2 lần
Lớp hiếm nhất 'Trend' chỉ chiếm 0.52% dữ liệu


### 1.1. Vì sao bài này khác hẳn hai bài trước

Ở Bài 1 và Bài 2, "chuỗi 1D" là một sự sắp đặt tuỳ tiện các cột trong file CSV.
Ở đây, chuỗi là một **câu tiếng Anh thật**, nơi thứ tự từ mang ý nghĩa:

- `[not, good]` và `[very, good]` chỉ khác một từ nhưng ngược nghĩa;
- `[runs, small]` là một cụm có nghĩa riêng, khác hẳn hai từ rời rạc.

Một kernel dài 3 trượt trên chuỗi token chính là một **bộ dò n-gram học được**.
Đây là trường hợp mà **tính cục bộ là có thật**, nên tích chập có cơ sở.

In [3]:
# ============================================================================
# KHỐI 3 — Hình 1: bốn góc nhìn EDA cho dữ liệu văn bản
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))

d = dist.reindex(CLASSES)
bars = axes[0, 0].bar([CLASS_VI[c] for c in CLASSES], d.values, color="#6366f1", width=0.6)
for b, v in zip(bars, d.values):
    axes[0, 0].text(b.get_x() + b.get_width() / 2, v + 120,
                    f"{v}\n({100*v/len(df):.1f}%)", ha="center", fontsize=8,
                    fontweight="bold")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title(f"(a) Phân bố nhãn — mất cân bằng {d.max()/d.min():.0f}:1 (thang log)",
                     fontweight="bold")
axes[0, 0].set_ylabel("Số đánh giá (log)")
axes[0, 0].tick_params(axis="x", rotation=22, labelsize=8)

wlen = df["Review Text"].str.split().str.len()
axes[0, 1].hist(wlen, bins=50, color="#10b981", edgecolor="white")
axes[0, 1].axvline(wlen.median(), color="#111827", ls="--",
                   label=f"Trung vị = {wlen.median():.0f} từ")
axes[0, 1].axvline(wlen.quantile(0.95), color="#ef4444", ls="--",
                   label=f"Bách phân vị 95 = {wlen.quantile(0.95):.0f} từ")
axes[0, 1].set_title("(b) Độ dài đánh giá — quyết định MAX_LEN", fontweight="bold")
axes[0, 1].set_xlabel("Số từ")
axes[0, 1].set_ylabel("Số đánh giá")
axes[0, 1].legend(fontsize=8)

sns.boxplot(data=df.assign(_len=wlen), x="Department Name", y="_len",
            order=CLASSES, ax=axes[1, 0], palette="viridis",
            hue="Department Name", legend=False, showfliers=False)
axes[1, 0].set_title("(c) Độ dài đánh giá theo lớp", fontweight="bold")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("Số từ")
axes[1, 0].set_xticks(range(len(CLASSES)))
axes[1, 0].set_xticklabels([CLASS_VI[c] for c in CLASSES], rotation=22, fontsize=8)

sns.boxplot(data=df, x="Department Name", y="Rating", order=CLASSES, ax=axes[1, 1],
            palette="magma", hue="Department Name", legend=False)
axes[1, 1].set_title("(d) Điểm đánh giá theo lớp — khá giống nhau,\n"
                     "nên nhãn phải suy ra từ NỘI DUNG chứ không từ điểm",
                     fontweight="bold", fontsize=9.5)
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Rating")
axes[1, 1].set_xticks(range(len(CLASSES)))
axes[1, 1].set_xticklabels([CLASS_VI[c] for c in CLASSES], rotation=22, fontsize=8)

plt.tight_layout()
plt.savefig(FIG / "c3_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_8908\2192656645.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Tiền xử lý văn bản: từ câu chữ sang chuỗi chỉ số

CNN không nhận chữ, nó nhận số. Quy trình gồm ba bước, và **cả ba phải tái lập
được nguyên văn bằng JavaScript** trên trình duyệt:

1. **Tách token** — viết thường, lấy mọi cụm `[a-z0-9']+` bằng biểu thức chính
   quy. Cố ý chọn quy tắc đơn giản nhất có thể để bản JS không lệch bản Python.
2. **Xây từ điển** — giữ `VOCAB_SIZE` token xuất hiện nhiều nhất **trên tập
   train**. Chỉ số 0 dành cho `<PAD>`, chỉ số 1 cho `<UNK>`.
3. **Đệm / cắt** về đúng `MAX_LEN` token.

> **Chống rò rỉ:** từ điển chỉ được đếm trên tập **train**. Nếu đếm trên toàn bộ
> dữ liệu thì thông tin tập test đã rò rỉ vào mô hình.

In [4]:
# ============================================================================
# KHỐI 4 — Tách token, chia dữ liệu, xây từ điển CHỈ trên tập train
# ============================================================================
VOCAB_SIZE = 4000
MAX_LEN = 100
PAD, UNK = 0, 1

TOKEN_RE = re.compile(r"[a-z0-9']+")


def tokenize(text):
    """Viết thường rồi lấy mọi cụm [a-z0-9']+. Bản JS dùng đúng regex này."""
    return TOKEN_RE.findall(str(text).lower())


texts = df["Review Text"].astype(str).values
labels = np.array([CLASSES.index(c) for c in df["Department Name"]])

idx_tmp, idx_test = train_test_split(
    np.arange(len(texts)), test_size=0.15, random_state=SEED, stratify=labels)
idx_train, idx_val = train_test_split(
    idx_tmp, test_size=0.1765, random_state=SEED, stratify=labels[idx_tmp])

counter = Counter()
for i in idx_train:
    counter.update(tokenize(texts[i]))

vocab = ["<PAD>", "<UNK>"] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}
print(f"Số token khác nhau trong tập train : {len(counter):,}")
print(f"Giữ lại từ điển                    : {len(vocab):,} (gồm <PAD>, <UNK>)")
covered = sum(counter[w] for w in vocab[2:])
print(f"Độ phủ token của từ điển           : {100*covered/sum(counter.values()):.2f}%")
print(f"\n20 token phổ biến nhất: {vocab[2:22]}")


def encode(text, max_len=MAX_LEN):
    """Chuỗi chữ -> mảng chỉ số dài max_len (cắt hoặc đệm 0 ở cuối)."""
    ids = [word2idx.get(t, UNK) for t in tokenize(text)][:max_len]
    return ids + [PAD] * (max_len - len(ids))


X_all = np.array([encode(t) for t in texts], dtype=np.int64)
Xtr, Xva, Xte = X_all[idx_train], X_all[idx_val], X_all[idx_test]
ytr_i, yva_i, yte_i = labels[idx_train], labels[idx_val], labels[idx_test]

print(f"\nTrain {Xtr.shape} | Val {Xva.shape} | Test {Xte.shape}")
print(f"Tỉ lệ token bị cắt (đánh giá dài hơn {MAX_LEN} từ): "
      f"{100*np.mean(df['Review Text'].str.split().str.len() > MAX_LEN):.2f}%")

demo = texts[idx_train[0]]
print(f"\nVí dụ mã hoá:\n  văn bản : {demo[:90]}...")
print(f"  token   : {tokenize(demo)[:12]}")
print(f"  chỉ số  : {encode(demo)[:12]} ...")

Số token khác nhau trong tập train : 12,622
Giữ lại từ điển                    : 4,000 (gồm <PAD>, <UNK>)
Độ phủ token của từ điển           : 98.45%

20 token phổ biến nhất: ['the', 'i', 'and', 'a', 'it', 'is', 'this', 'to', 'in', 'but', 'on', 'for', 'of', 'with', 'was', 'so', 'my', 'dress', 'not', 'that']



Train (15838, 100) | Val (3395, 100) | Test (3395, 100)
Tỉ lệ token bị cắt (đánh giá dài hơn 100 từ): 6.45%

Ví dụ mã hoá:
  văn bản : I was looking for a classic, white t-shirt that wasn't too big (i.e. a tunic) or too fitte...
  token   : ['i', 'was', 'looking', 'for', 'a', 'classic', 'white', 't', 'shirt', 'that', "wasn't", 'too']
  chỉ số  : [3, 16, 184, 13, 5, 526, 139, 302, 83, 21, 282, 36] ...


In [5]:
# ============================================================================
# KHỐI 5 — One-hot nhãn và trọng số lớp chống mất cân bằng
# Trọng số = N / (C * n_c): lớp hiếm được nhân hệ số lớn nên không bị bỏ qua.
# ============================================================================
def one_hot_np(y, n):
    out = np.zeros((len(y), n))
    out[np.arange(len(y)), y] = 1.0
    return out


ytr = one_hot_np(ytr_i, N_CLASS)
yva = one_hot_np(yva_i, N_CLASS)
yte = one_hot_np(yte_i, N_CLASS)

counts = np.bincount(ytr_i, minlength=N_CLASS).astype(float)
class_weight = len(ytr_i) / (N_CLASS * counts)
print("Trọng số lớp (tính trên tập train):")
for c, n, w in zip(CLASSES, counts, class_weight):
    print(f"  {c:<10} n = {int(n):5d}   trọng số = {w:6.2f}")

Trọng số lớp (tính trên tập train):
  Bottoms    n =  2563   trọng số =   1.03
  Dresses    n =  4301   trọng số =   0.61
  Intimate   n =  1157   trọng số =   2.28
  Jackets    n =   702   trọng số =   3.76
  Tops       n =  7033   trọng số =   0.38
  Trend      n =    82   trọng số =  32.19


## 3. Cài đặt CNN 1D thuần NumPy (có tầng Embedding)

In [6]:
import numpy as np

# ============================================================================
# 1. HÀM KÍCH HOẠT
# ============================================================================


def relu(z):
    """f(z) = max(0, z)."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0. Tại z = 0 ta quy ước đạo hàm bằng 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để exp() không tràn số."""
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ============================================================================
# 2. TÍCH CHẬP 1D Ở MỨC HÀM
# ============================================================================
#
# Định nghĩa toán học dùng trong báo cáo (tích chập 'valid', stride 1):
#
#     z[n, o, l] = sum_{c=0}^{C_in-1} sum_{k=0}^{K-1} W[o, c, k] * x[n, c, l+k] + b[o]
#
# Bản `conv1d_naive` viết đúng ba vòng lặp như công thức trên — dễ đọc, dùng để
# kiểm chứng. Bản `conv1d_forward` dùng thủ thuật im2col nên nhanh hơn hàng chục
# lần mà vẫn cho kết quả giống hệt (notebook có assert đối chiếu hai bản).


def conv1d_naive(X, W, b):
    """Tích chập theo đúng định nghĩa toán học — ba vòng lặp tường minh.

    Chậm, nhưng là bản "nguyên văn công thức" để đối chiếu với bản im2col.
    """
    N, C_in, L_in = X.shape
    C_out, C_in_w, K = W.shape
    assert C_in == C_in_w, f"Số kênh vào không khớp: X có {C_in}, W cần {C_in_w}"
    L_out = L_in - K + 1

    Z = np.zeros((N, C_out, L_out))
    for o in range(C_out):          # từng bộ lọc
        for l in range(L_out):      # từng vị trí cửa sổ trượt
            for k in range(K):      # từng ô trong kernel
                Z[:, o, l] += np.sum(W[o, :, k] * X[:, :, l + k], axis=1)
            Z[:, o, l] += b[o]
    return Z


def im2col_1d(X, K):
    """Dàn mọi cửa sổ trượt thành một ma trận để biến tích chập thành nhân ma trận.

    Trả về `cols` có shape (N, C_in * K, L_out) với quy ước chỉ số:

        cols[n, c * K + k, l] = X[n, c, l + k]

    Nhờ layout `c * K + k`, mảng W shape (C_out, C_in, K) chỉ cần `reshape`
    (không cần transpose) là khớp: W.reshape(C_out, C_in*K)[o, c*K+k] = W[o,c,k].
    """
    N, C_in, L_in = X.shape
    L_out = L_in - K + 1
    cols = np.empty((N, C_in * K, L_out), dtype=X.dtype)
    for k in range(K):
        # Bước nhảy K: chỉ số k, K+k, 2K+k, ... đúng là c*K+k với c = 0, 1, 2, ...
        cols[:, k::K, :] = X[:, :, k:k + L_out]
    return cols


def col2im_1d(dcols, x_shape, K):
    """Phép chuyển vị của im2col: cộng dồn gradient về đúng vị trí trong X.

    Một phần tử x[n, c, i] tham gia nhiều cửa sổ khác nhau, nên gradient của nó
    là TỔNG các đóng góp — đây chính là chỗ phép cộng dồn (`+=`) là bắt buộc.
    """
    N, C_in, L_in = x_shape
    L_out = L_in - K + 1
    dX = np.zeros(x_shape, dtype=float)
    for k in range(K):
        dX[:, :, k:k + L_out] += dcols[:, k::K, :]
    return dX


def conv1d_forward(X, W, b):
    """Tích chập 'valid' stride 1 bằng im2col. Trả về (Z, cache)."""
    C_out, C_in, K = W.shape
    cols = im2col_1d(X, K)                              # (N, C_in*K, L_out)
    Wr = W.reshape(C_out, C_in * K)                     # (C_out, C_in*K)
    Z = np.einsum("ok,nkl->nol", Wr, cols, optimize=True) + b[None, :, None]
    return Z, (cols, X.shape, W.shape)


def conv1d_backward(dZ, W, cache):
    """Lan truyền ngược qua tích chập. Trả về (dX, dW, db).

    Cho dZ = dL/dZ shape (N, C_out, L_out), ba công thức suy ra trực tiếp từ
    z = W·cols + b:

        dW[o, c, k] = sum_{n,l} dZ[n,o,l] * cols[n, c*K+k, l]
        db[o]       = sum_{n,l} dZ[n,o,l]
        dcols       = W^T · dZ   ->  dX = col2im(dcols)

    Chú ý db là tổng trên CẢ batch VÀ mọi vị trí l: một bias duy nhất được dùng
    lại ở mọi vị trí (chia sẻ trọng số), nên gradient của nó gom hết đóng góp.
    Điều tương tự xảy ra với dW — đây chính là dấu vết toán học của việc chia sẻ
    trọng số, và là điểm khác biệt cốt lõi so với tầng Dense.
    """
    cols, x_shape, w_shape = cache
    C_out, C_in, K = w_shape

    dW = np.einsum("nol,nkl->ok", dZ, cols, optimize=True).reshape(w_shape)
    db = dZ.sum(axis=(0, 2))

    Wr = W.reshape(C_out, C_in * K)
    dcols = np.einsum("ok,nol->nkl", Wr, dZ, optimize=True)
    dX = col2im_1d(dcols, x_shape, K)
    return dX, dW, db


# ============================================================================
# 3. CÁC LỚP TẦNG
# ============================================================================


class Layer:
    """Giao diện chung. `params()` trả về list [(tên, mảng_trọng_số, hàm_gán)]."""

    trainable = False

    def forward(self, X, training=False):
        raise NotImplementedError

    def backward(self, dOut):
        raise NotImplementedError

    def param_list(self):
        """Trả về list các mảng tham số (để Adam cập nhật tại chỗ)."""
        return []

    def grad_list(self):
        return []

    def describe(self, in_shape):
        """(mô tả, out_shape, số tham số) — dùng để in bảng đặc tả tensor."""
        return (type(self).__name__, in_shape, 0)

    def to_dict(self, decimals):
        return {"type": type(self).__name__.lower()}


class Conv1D(Layer):
    """Tầng tích chập 1D: (N, C_in, L) -> (N, C_out, L-K+1).

    Khởi tạo He (Kaiming) với fan_in = C_in * K, phù hợp với ReLU đứng sau.
    """

    trainable = True

    def __init__(self, c_in, c_out, k, rng):
        self.c_in, self.c_out, self.k = c_in, c_out, k
        fan_in = c_in * k
        self.W = rng.normal(0.0, np.sqrt(2.0 / fan_in), (c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        Z, self._cache = conv1d_forward(X, self.W, self.b)
        return Z

    def backward(self, dZ):
        dX, self.dW, self.db = conv1d_backward(dZ, self.W, self._cache)
        return dX

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        C, L = in_shape
        out = (self.c_out, L - self.k + 1)
        n = self.c_out * self.c_in * self.k + self.c_out
        return (f"Conv1D(C {self.c_in}→{self.c_out}, K={self.k})", out, n)

    def to_dict(self, decimals):
        return {
            "type": "conv1d",
            "c_in": self.c_in, "c_out": self.c_out, "k": self.k,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class ReLU(Layer):
    def forward(self, X, training=False):
        self._z = X
        return relu(X)

    def backward(self, dOut):
        return dOut * relu_grad(self._z)

    def describe(self, in_shape):
        return ("ReLU", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "relu"}


class MaxPool1D(Layer):
    """Gom cụm cực đại theo cửa sổ không chồng lấn, size = stride = `p`.

    Nếu L không chia hết cho p, phần dư ở cuối bị bỏ (floor) — giống hành vi
    mặc định của nn.MaxPool1d và MaxPooling1D.
    """

    def __init__(self, p=2):
        self.p = p

    def forward(self, X, training=False):
        N, C, L = X.shape
        p = self.p
        L_out = L // p
        Xc = X[:, :, :L_out * p].reshape(N, C, L_out, p)
        self._argmax = Xc.argmax(axis=3)
        self._shape = X.shape
        return Xc.max(axis=3)

    def backward(self, dOut):
        N, C, L = self._shape
        p = self.p
        L_out = L // p
        dX = np.zeros((N, C, L_out * p))
        # Chỉ ô thắng (đạt cực đại) nhận gradient, các ô còn lại nhận 0.
        n_i, c_i, l_i = np.ogrid[:N, :C, :L_out]
        dXc = dX.reshape(N, C, L_out, p)
        dXc[n_i, c_i, l_i, self._argmax] = dOut
        full = np.zeros(self._shape)
        full[:, :, :L_out * p] = dXc.reshape(N, C, L_out * p)
        return full

    def describe(self, in_shape):
        C, L = in_shape
        return (f"MaxPool1D(p={self.p})", (C, L // self.p), 0)

    def to_dict(self, decimals):
        return {"type": "maxpool1d", "p": self.p}


class GlobalMaxPool1D(Layer):
    """Lấy giá trị lớn nhất trên toàn trục thời gian: (N, C, L) -> (N, C).

    Dùng cho văn bản: mỗi bộ lọc trả lời "mẫu cục bộ mà tôi phụ trách có xuất
    hiện ở đâu đó trong câu hay không", nên độ dài câu không còn ảnh hưởng tới
    số chiều đầu ra — đó là cách CNN xử lý câu dài ngắn khác nhau.
    """

    def forward(self, X, training=False):
        self._argmax = X.argmax(axis=2)
        self._shape = X.shape
        return X.max(axis=2)

    def backward(self, dOut):
        N, C, L = self._shape
        dX = np.zeros(self._shape)
        n_i, c_i = np.ogrid[:N, :C]
        dX[n_i, c_i, self._argmax] = dOut
        return dX

    def describe(self, in_shape):
        C, L = in_shape
        return ("GlobalMaxPool1D", (C,), 0)

    def to_dict(self, decimals):
        return {"type": "globalmaxpool1d"}


class Flatten(Layer):
    """(N, C, L) -> (N, C*L). Thứ tự dàn phẳng là C trước, L sau (C-order)."""

    def forward(self, X, training=False):
        self._shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dOut):
        return dOut.reshape(self._shape)

    def describe(self, in_shape):
        return ("Flatten", (int(np.prod(in_shape)),), 0)

    def to_dict(self, decimals):
        return {"type": "flatten"}


class Dense(Layer):
    """Tầng kết nối đầy đủ: (N, D) -> (N, M), W shape (D, M)."""

    trainable = True

    def __init__(self, d_in, d_out, rng):
        self.d_in, self.d_out = d_in, d_out
        self.W = rng.normal(0.0, np.sqrt(2.0 / d_in), (d_in, d_out))
        self.b = np.zeros(d_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        self._x = X
        return X @ self.W + self.b

    def backward(self, dOut):
        self.dW = self._x.T @ dOut
        self.db = dOut.sum(axis=0)
        return dOut @ self.W.T

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        n = self.d_in * self.d_out + self.d_out
        return (f"Dense({self.d_in}→{self.d_out})", (self.d_out,), n)

    def to_dict(self, decimals):
        return {
            "type": "dense",
            "d_in": self.d_in, "d_out": self.d_out,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class Dropout(Layer):
    """Inverted dropout — chia cho keep_prob ngay lúc train nên lúc suy luận
    không phải chỉnh gì, trọng số xuất ra JSON dùng trực tiếp được."""

    def __init__(self, rate, rng):
        self.rate = rate
        self.rng = rng

    def forward(self, X, training=False):
        if not training or self.rate <= 0.0:
            self._mask = None
            return X
        keep = 1.0 - self.rate
        self._mask = (self.rng.random(X.shape) < keep) / keep
        return X * self._mask

    def backward(self, dOut):
        return dOut if self._mask is None else dOut * self._mask

    def describe(self, in_shape):
        return (f"Dropout(p={self.rate})", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "dropout", "rate": self.rate}


class Embedding(Layer):
    """Tra bảng nhúng cho văn bản: (N, L) chỉ số nguyên -> (N, C_emb, L).

    Đầu ra đã ở dạng channels-first để Conv1D dùng trực tiếp. Chỉ số 0 được
    dành riêng cho token đệm (PAD) và vector của nó bị ghim bằng 0 (cả trong
    khởi tạo lẫn sau mỗi bước cập nhật), nên phần đệm không đóng góp gì vào
    tích chập.
    """

    trainable = True

    def __init__(self, vocab_size, dim, rng, pad_idx=0):
        self.vocab_size, self.dim, self.pad_idx = vocab_size, dim, pad_idx
        self.E = rng.normal(0.0, 0.05, (vocab_size, dim))
        self.E[pad_idx] = 0.0
        self.dE = np.zeros_like(self.E)

    def forward(self, idx, training=False):
        self._idx = idx.astype(np.int64)
        # (N, L, dim) -> (N, dim, L)
        return self.E[self._idx].transpose(0, 2, 1)

    def backward(self, dOut):
        # dOut: (N, dim, L) -> (N, L, dim), rồi cộng dồn theo chỉ số token.
        g = dOut.transpose(0, 2, 1)
        self.dE = np.zeros_like(self.E)
        # np.add.at cộng dồn đúng khi một token xuất hiện nhiều lần trong batch
        # (phép gán thường sẽ ghi đè và làm mất gradient).
        np.add.at(self.dE, self._idx.ravel(), g.reshape(-1, self.dim))
        self.dE[self.pad_idx] = 0.0
        return None  # Embedding là tầng đầu tiên — không cần truyền ngược nữa

    def param_list(self):
        return [self.E]

    def grad_list(self):
        return [self.dE]

    def describe(self, in_shape):
        (L,) = in_shape
        return (f"Embedding({self.vocab_size}→{self.dim})", (self.dim, L),
                self.vocab_size * self.dim)

    def to_dict(self, decimals):
        return {
            "type": "embedding",
            "vocab_size": self.vocab_size, "dim": self.dim, "pad_idx": self.pad_idx,
            "E": np.round(self.E, decimals).tolist(),
        }


# ============================================================================
# 4. BỘ CHỨA TUẦN TỰ + HUẤN LUYỆN
# ============================================================================


class CNN1D:
    """CNN 1D tuần tự, tự cài forward/backward/Adam.

    `task` quyết định đầu ra và hàm mất mát:
        binary      : 1 nơ-ron + Sigmoid  + Binary Cross-Entropy
        regression  : 1 nơ-ron + Linear   + Mean Squared Error
        multiclass  : C nơ-ron + Softmax  + Categorical Cross-Entropy

    Với cả ba, đạo hàm của loss theo pre-activation cuối rút gọn về (ŷ − y) —
    đó chính là lý do ta ghép Sigmoid/Softmax với Cross-Entropy và Linear
    với MSE, chi tiết ở Chương I của báo cáo.
    """

    def __init__(self, layers, task="binary", lr=1e-3, l2=0.0, seed=42,
                 class_weight=None, clip_norm=None):
        assert task in {"binary", "regression", "multiclass"}
        self.layers = layers
        self.task = task
        self.lr = lr
        self.l2 = l2
        self.clip_norm = clip_norm
        self.class_weight = None if class_weight is None else np.asarray(class_weight, float)
        self.rng = np.random.default_rng(seed)

        # Trạng thái Adam: một cặp (m, v) cho mỗi mảng tham số.
        self._params = [p for L in layers for p in L.param_list()]
        self.m = [np.zeros_like(p) for p in self._params]
        self.v = [np.zeros_like(p) for p in self._params]
        self.t = 0
        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training=False):
        A = X
        for L in self.layers:
            A = L.forward(A, training=training)
        Z = A                                  # pre-activation của head
        if self.task == "binary":
            return sigmoid(Z), Z
        if self.task == "multiclass":
            return softmax(Z), Z
        return Z, Z

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true, logits=None):
        """Hàm mất mát. Nếu truyền `logits` (pre-activation của head) thì dùng dạng
        log-sum-exp / log-sigmoid.

        Vì sao phải quan tâm: cách viết "ngây thơ" `-log(p + eps)` gài một sai số
        hệ thống bằng eps/p. Khi mạng gán cho lớp đúng một xác suất rất nhỏ
        (p ~ 1e-8, hay gặp ở epoch đầu), sai số đó lên tới ~1e-4 tương đối và
        KHÔNG khớp với gradient giải tích (ŷ − y) — đủ để phép kiểm tra gradient
        bằng sai phân số thất bại dù backward hoàn toàn đúng.

        Dạng logit dưới đây không cần eps, ổn định số học, và nhất quán tuyệt đối
        với dZ = ŷ − y. Đây cũng chính là lý do PyTorch khuyên dùng
        `binary_cross_entropy_with_logits` thay cho `sigmoid` rồi `log`.
        """
        n = y_true.shape[0]
        eps = 1e-12
        if self.task == "binary":
            if logits is not None:
                z = logits
                # max(z,0) − z·y + log(1 + exp(−|z|)) — BCE-with-logits ổn định.
                base = float(np.mean(np.maximum(z, 0.0) - z * y_true
                                     + np.log1p(np.exp(-np.abs(z)))))
            else:
                base = -np.mean(y_true * np.log(y_pred + eps)
                                + (1 - y_true) * np.log(1 - y_pred + eps))
        elif self.task == "multiclass":
            if logits is not None:
                z = logits - logits.max(axis=1, keepdims=True)
                log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
                per = -np.sum(y_true * log_p, axis=1, keepdims=True)
            else:
                per = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per if w is None else per * w) / n)
        else:
            base = float(np.mean((y_pred - y_true) ** 2))
        reg = self.l2 * sum(np.sum(p * p) for p in self._params) / (2 * n) if self.l2 else 0.0
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, y_pred, y_true):
        n = y_true.shape[0]
        dZ = (y_pred - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        d = dZ
        for L in reversed(self.layers):
            d = L.backward(d)
            if d is None:       # đã tới tầng Embedding
                break

        if self.l2:
            for L in self.layers:
                if L.trainable:
                    for p, g in zip(L.param_list(), L.grad_list()):
                        g += self.l2 * p / n

    # ------------------------------ Adam -----------------------------------
    def _adam(self, beta1=0.9, beta2=0.999, eps=1e-8):
        grads = [g for L in self.layers for g in L.grad_list()]

        if self.clip_norm:
            # Cắt chuẩn gradient toàn cục: giữ hướng, chỉ co độ dài. Bài văn bản
            # có gradient nhảy vọt khi một n-gram hiếm xuất hiện, nên cần thứ này.
            total = np.sqrt(sum(float(np.sum(g * g)) for g in grads))
            if total > self.clip_norm:
                scale = self.clip_norm / (total + 1e-12)
                grads = [g * scale for g in grads]

        self.t += 1
        for i, (p, g) in enumerate(zip(self._params, grads)):
            self.m[i] = beta1 * self.m[i] + (1 - beta1) * g
            self.v[i] = beta2 * self.v[i] + (1 - beta2) * (g * g)
            mhat = self.m[i] / (1 - beta1 ** self.t)
            vhat = self.v[i] / (1 - beta2 ** self.t)
            p -= self.lr * mhat / (np.sqrt(vhat) + eps)   # cập nhật TẠI CHỖ

        # Giữ vector PAD bằng 0 sau mỗi bước (Adam có thể đẩy nó lệch khỏi 0).
        for L in self.layers:
            if isinstance(L, Embedding):
                L.E[L.pad_idx] = 0.0

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = float(np.sum((y - p) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        return 1.0 - ss_res / ss_tot            # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=100, batch_size=32,
            verbose_every=10, patience=None, eval_batch=512, eval_subset=None,
            val_score_fn=None):
        """Chu trình 4 bước mỗi mini-batch: Forward → Loss → Backward → Update.

        `eval_subset`: nếu đặt, loss/metric TRAIN mỗi epoch chỉ tính trên một mẫu
        con cố định cỡ này (chọn một lần, không đổi giữa các epoch) thay vì toàn
        bộ tập train. Chỉ dùng cho bài văn bản 23k mẫu, nơi một lượt forward đầy
        đủ mỗi epoch đắt hơn cả việc huấn luyện. Early stopping vẫn dựa trên
        validation đầy đủ nên không ảnh hưởng tới việc chọn mô hình.

        `val_score_fn(y_true_onehot, y_prob) -> float`: nếu đặt, early stopping
        và việc giữ trọng số tốt nhất sẽ **cực đại hoá** điểm này thay vì cực
        tiểu hoá val_loss.

        Vì sao cần: với dữ liệu mất cân bằng nặng và cross-entropy CÓ TRỌNG SỐ
        LỚP, val_loss dao động mạnh và đạt cực tiểu rất sớm, trong khi Macro-F1
        vẫn còn đang lên. Chọn mô hình theo val_loss khi đó dừng quá sớm và cho
        mô hình kém hẳn. Tiêu chí chọn mô hình phải là chỉ số ta thật sự quan tâm.
        """
        n = X.shape[0]
        # Quy ước nội bộ: luôn CỰC TIỂU `best_val`. Khi dùng val_score_fn ta lấy
        # dấu âm của điểm số, nên một nhánh early-stopping duy nhất phục vụ cả hai.
        best_val, best_state, wait = np.inf, None, 0
        self.history.setdefault("val_score", [])

        if eval_subset is not None and eval_subset < n:
            sub = self.rng.choice(n, size=eval_subset, replace=False)
            X_tr_eval, y_tr_eval = X[sub], y[sub]
        else:
            X_tr_eval, y_tr_eval = X, y

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            for s in range(0, n, batch_size):
                sl = idx[s:s + batch_size]
                xb, yb = X[sl], y[sl]
                yp, _ = self.forward(xb, training=True)     # (1) Forward
                self.backward(yp, yb)                      # (3) Backward
                self._adam()                               # (4) Update

            tr_pred, tr_logit = self._eval(X_tr_eval, batch=eval_batch)
            tr_loss = self.loss(tr_pred, y_tr_eval, logits=tr_logit)   # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X_tr_eval, y_tr_eval, eval_batch))

            if X_val is not None:
                va_pred, va_logit = self._eval(X_val, batch=eval_batch)
                va_loss = self.loss(va_pred, y_val, logits=va_logit)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val, eval_batch))

                if val_score_fn is not None:
                    score = float(val_score_fn(y_val, va_pred))
                    self.history["val_score"].append(score)
                    watched, label = -score, "val_score"
                else:
                    watched, label = va_loss, "val_loss"

                if patience is not None:
                    if watched < best_val - 1e-6:
                        best_val, wait = watched, 0
                        best_state = [p.copy() for p in self._params]
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                shown = -best_val if val_score_fn is not None else best_val
                                print(f"  ⏹ Early stopping tại epoch {ep} "
                                      f"({label} tốt nhất = {shown:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = (f"  epoch {ep:4d} | train_loss={tr_loss:.4f} "
                       f"| train_metric={self.history['train_metric'][-1]:.4f}")
                if X_val is not None:
                    msg += (f" | val_loss={self.history['val_loss'][-1]:.4f}"
                            f" | val_metric={self.history['val_metric'][-1]:.4f}")
                    if val_score_fn is not None:
                        msg += f" | val_score={self.history['val_score'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            # Trả tham số về trạng thái tốt nhất trên validation, ghi TẠI CHỖ để
            # self._params (và các layer) vẫn trỏ tới cùng mảng.
            for p, bp in zip(self._params, best_state):
                p[...] = bp
        return self

    # ---------------------------- inference --------------------------------
    def _eval(self, X, batch=512):
        """Suy luận theo lô, trả về (xác suất, logits). Logits cần cho `loss()`."""
        ps, zs = [], []
        for s in range(0, X.shape[0], batch):
            p, z = self.forward(X[s:s + batch], training=False)
            ps.append(p)
            zs.append(z)
        return np.concatenate(ps, axis=0), np.concatenate(zs, axis=0)

    def predict_proba(self, X, batch=512):
        return self._eval(X, batch)[0]

    def predict(self, X, threshold=0.5, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return (p >= threshold).astype(int)
        if self.task == "multiclass":
            return p.argmax(1)
        return p

    # ------------------------- đặc tả và xuất JSON -------------------------
    def n_params(self):
        return int(sum(p.size for p in self._params))

    def shape_table(self, in_shape):
        """Bảng (tầng, đầu vào, đầu ra, số tham số) — phần đặc tả tensor của báo cáo."""
        rows, shape = [], tuple(in_shape)
        for L in self.layers:
            name, out, n = L.describe(shape)
            rows.append({"Tầng": name,
                         "Input shape": f"(batch, {', '.join(map(str, shape))})",
                         "Output shape": f"(batch, {', '.join(map(str, out))})",
                         "Số tham số": n,
                         "Huấn luyện được": "✓" if L.trainable else "—"})
            shape = out
        return rows

    def to_dict(self, decimals=6):
        head = {"binary": "sigmoid", "multiclass": "softmax", "regression": "linear"}[self.task]
        return {
            "kind": "cnn1d",
            "task": self.task,
            "head": head,
            "n_params": self.n_params(),
            "n_trainable_layers": sum(1 for L in self.layers if L.trainable),
            "layers": [L.to_dict(decimals) for L in self.layers],
        }


# ============================================================================
# 5. TIỆN ÍCH
# ============================================================================


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(bundle, x_single):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript.

    Nhận đúng dict đã ghi ra `model_cnn.json` (nên mọi trọng số đã bị làm tròn)
    và MỘT mẫu. Dùng trong notebook để kiểm tra parity NumPy ↔ JSON ↔ JS: nếu
    hàm này khớp với mô hình gốc thì bản JS chỉ cần dịch nguyên văn là đúng.

    x_single:
        - bài bảng  : mảng (C_in, L) hoặc (L,) — sẽ được thêm trục kênh
        - bài văn bản: mảng (L,) chỉ số token nguyên
    """
    a = np.asarray(x_single)
    layers = bundle["layers"]

    if layers[0]["type"] == "embedding":
        a = a.astype(np.int64)[None, :]                    # (1, L)
    else:
        if a.ndim == 1:
            a = a[None, :]                                 # (C_in=1, L)
        a = a.astype(float)[None, ...]                     # (1, C_in, L)

    for layer in layers:
        t = layer["type"]
        if t == "embedding":
            E = np.asarray(layer["E"])
            a = E[a].transpose(0, 2, 1)
        elif t == "conv1d":
            W = np.asarray(layer["W"])
            b = np.asarray(layer["b"])
            a, _ = conv1d_forward(a, W, b)
        elif t == "relu":
            a = relu(a)
        elif t == "maxpool1d":
            p = layer["p"]
            N, C, L = a.shape
            L_out = L // p
            a = a[:, :, :L_out * p].reshape(N, C, L_out, p).max(axis=3)
        elif t == "globalmaxpool1d":
            a = a.max(axis=2)
        elif t == "flatten":
            a = a.reshape(a.shape[0], -1)
        elif t == "dense":
            a = a @ np.asarray(layer["W"]) + np.asarray(layer["b"])
        elif t == "dropout":
            pass                                           # suy luận: dropout vô hiệu
        else:
            raise ValueError(f"Tầng lạ trong bundle: {t}")

    head = bundle["head"]
    if head == "sigmoid":
        return float(sigmoid(a).ravel()[0])
    if head == "softmax":
        return softmax(a.reshape(1, -1))[0]
    return float(a.ravel()[0])

### 3.1. Kiến trúc

Theo đúng sơ đồ slide 11: `tokens → embedding → Conv1D → ReLU → Pooling → Dense
→ classification`, mở rộng thành phiên bản sâu của slide 16.

$$\text{tokens} \to \text{Embedding} \to \text{Conv}_1 \to \text{Conv}_2 \to \text{Pool} \to \text{Conv}_3 \to \text{GlobalMaxPool} \to \text{Dense}_1 \to \text{Dense}_2 \to \text{Softmax}$$

Hai điểm kỹ thuật đáng chú ý:

- **`GlobalMaxPool1D`** lấy giá trị lớn nhất trên **toàn trục thời gian**. Mỗi
  bộ lọc vì thế trả lời câu hỏi *"mẫu n-gram mà tôi phụ trách có xuất hiện ở
  đâu đó trong câu không?"* — nên câu dài hay ngắn đều cho ra vector cùng số
  chiều. Đây là cách CNN xử lý độ dài thay đổi.
- **Vector `<PAD>` bị ghim bằng 0** cả lúc khởi tạo lẫn sau mỗi bước Adam, nên
  phần đệm không đóng góp gì vào tích chập.

**Đếm tầng:** theo quy ước đã nêu ở Bài 1 (chỉ đếm phép biến đổi có tham số
huấn luyện được), mạng này có **6 tầng**: 1 Embedding + 3 Conv1D + 2 Dense.
Riêng phần *ngăn xếp tích chập/dense* vẫn đúng 5 tầng như slide 16; Embedding là
một **bảng tra cứu học được**, ta tách riêng để bảng đặc tả minh bạch.

In [7]:
# ============================================================================
# KHỐI 7 — Dựng CNN văn bản và in bảng đặc tả tensor
# Đường đi độ dài chuỗi: 100 -> 98 -> 96 -> (pool) 48 -> 46 -> (global) 1
# ============================================================================
HP = dict(lr=1.5e-3, l2=0.0, dropout=0.25, batch_size=64, epochs=40, patience=10,
          emb_dim=16, f1=32, f2=32, f3=48, clip=5.0)
print("Siêu tham số (chọn trên tập VALIDATION theo Macro-F1):")
print(HP, "\n")


def build_cnn(seed=SEED):
    r = np.random.default_rng(seed)
    layers = [
        Embedding(VOCAB_SIZE, HP["emb_dim"], r, pad_idx=PAD),   # tầng 0 (bảng tra cứu)
        Conv1D(HP["emb_dim"], HP["f1"], 3, r), ReLU(),          # tầng 1: 3-gram
        Conv1D(HP["f1"], HP["f2"], 3, r), ReLU(),               # tầng 2: mẫu của mẫu
        MaxPool1D(2),
        Conv1D(HP["f2"], HP["f3"], 3, r), ReLU(),               # tầng 3
        GlobalMaxPool1D(),
        Dropout(HP["dropout"], r),
        Dense(HP["f3"], 32, r), ReLU(),                         # tầng 4
        Dense(32, N_CLASS, r),                                  # tầng 5 (+ Softmax)
    ]
    return CNN1D(layers=layers, task="multiclass", lr=HP["lr"], l2=HP["l2"],
                 seed=seed, class_weight=class_weight, clip_norm=HP["clip"])


cnn = build_cnn()
shape_table = pd.DataFrame(cnn.shape_table((MAX_LEN,)))
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", shape_table["Số tham số"].sum(), ""]
emb_params = VOCAB_SIZE * HP["emb_dim"]
print(f"Tổng tham số            : {cnn.n_params():,}")
print(f"  trong đó Embedding    : {emb_params:,} ({100*emb_params/cnn.n_params():.1f}%)")
print(f"  phần tích chập + dense: {cnn.n_params() - emb_params:,}")
print(f"Số tầng huấn luyện được : {cnn.to_dict()['n_trainable_layers']}")
shape_table

Siêu tham số (chọn trên tập VALIDATION theo Macro-F1):
{'lr': 0.0015, 'l2': 0.0, 'dropout': 0.25, 'batch_size': 64, 'epochs': 40, 'patience': 10, 'emb_dim': 16, 'f1': 32, 'f2': 32, 'f3': 48, 'clip': 5.0} 

Tổng tham số            : 75,094
  trong đó Embedding    : 64,000 (85.2%)
  phần tích chập + dense: 11,094
Số tầng huấn luyện được : 6


,Tầng,Input shape,Output shape,Số tham số,Huấn luyện được
0,Embedding(4000→16),"(batch, 100)","(batch, 16, 100)",64000,✓
1,"Conv1D(C 16→32, K=3)","(batch, 16, 100)","(batch, 32, 98)",1568,✓
2,ReLU,"(batch, 32, 98)","(batch, 32, 98)",0,—
3,"Conv1D(C 32→32, K=3)","(batch, 32, 98)","(batch, 32, 96)",3104,✓
4,ReLU,"(batch, 32, 96)","(batch, 32, 96)",0,—
5,MaxPool1D(p=2),"(batch, 32, 96)","(batch, 32, 48)",0,—
6,"Conv1D(C 32→48, K=3)","(batch, 32, 48)","(batch, 48, 46)",4656,✓
7,ReLU,"(batch, 48, 46)","(batch, 48, 46)",0,—
8,GlobalMaxPool1D,"(batch, 48, 46)","(batch, 48)",0,—
9,Dropout(p=0.25),"(batch, 48)","(batch, 48)",0,—


### 3.2. Kiểm chứng gradient — gồm cả tầng Embedding

Tầng Embedding có một cái bẫy: khi **một token xuất hiện nhiều lần** trong cùng
một batch, gradient của nó phải là **tổng** các đóng góp. Dùng phép gán thường
sẽ ghi đè và làm mất gradient; phải dùng `np.add.at` để cộng dồn. Phép kiểm tra
dưới đây bắt được đúng loại lỗi đó.

In [8]:
# ============================================================================
# KHỐI 8 — Gradient check trên kiến trúc văn bản thật
# ============================================================================
def pool_relu_pattern(net):
    pat = []
    for L in net.layers:
        if isinstance(L, (MaxPool1D, GlobalMaxPool1D)):
            pat.append(L._argmax.copy())
        elif isinstance(L, ReLU):
            pat.append(L._z > 0)
    return pat


def same_pattern(a, b):
    return len(a) == len(b) and all(np.array_equal(u, v) for u, v in zip(a, b))


def gradient_check(net, X, yy, n_coord=20, h=1e-5):
    p, z = net.forward(X, training=False)
    base = pool_relu_pattern(net)
    net.backward(p, yy)
    grads = [g.copy() for L in net.layers for g in L.grad_list()]
    params = [q for L in net.layers for q in L.param_list()]
    worst, tested, skipped = 0.0, 0, 0
    for q, g in zip(params, grads):
        flat = q.ravel()
        # Với bảng nhúng, chỉ những hàng ứng với token CÓ trong batch mới có
        # gradient khác 0; lấy mẫu ở đó mới kiểm tra được thứ có ý nghĩa.
        nz = np.flatnonzero(g.ravel())
        pool_idx = nz if nz.size else np.arange(flat.size)
        picks = pool_idx[np.linspace(0, pool_idx.size - 1,
                                     min(n_coord, pool_idx.size)).astype(int)]
        for i in picks:
            old = flat[i]
            flat[i] = old + h
            pp, zp = net.forward(X, training=False)
            lp, pat_p = net.loss(pp, yy, logits=zp), pool_relu_pattern(net)
            flat[i] = old - h
            pm, zm = net.forward(X, training=False)
            lm, pat_m = net.loss(pm, yy, logits=zm), pool_relu_pattern(net)
            flat[i] = old
            if not (same_pattern(base, pat_p) and same_pattern(base, pat_m)):
                skipped += 1
                continue
            num = (lp - lm) / (2 * h)
            ana = g.ravel()[i]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
            tested += 1
    return worst, tested, skipped


# Chọn một batch nhỏ CÓ token lặp lại để phép kiểm tra thực sự chạm vào np.add.at
check_net = build_cnn(seed=13)
Xc, yc = Xtr[:16], ytr[:16]
rep = Counter(Xc[Xc > UNK].tolist())
print(f"Batch kiểm tra có {sum(1 for v in rep.values() if v > 1)} token xuất hiện lặp "
      f"(nhiều nhất {max(rep.values())} lần) — đủ để kiểm tra phép cộng dồn.")
worst, tested, skipped = gradient_check(check_net, Xc, yc)
print(f"\nSai số tương đối lớn nhất : {worst:.3e}")
print(f"Số toạ độ đã kiểm tra     : {tested}  (bỏ {skipped} toạ độ ở nếp gấp)")
assert worst < 1e-6, "Gradient check FAILED"
print("\n✅ Gradient của CẢ tầng Embedding lẫn ngăn xếp tích chập là ĐÚNG.")

Batch kiểm tra có 102 token xuất hiện lặp (nhiều nhất 48 lần) — đủ để kiểm tra phép cộng dồn.



Sai số tương đối lớn nhất : 4.906e-08
Số toạ độ đã kiểm tra     : 146  (bỏ 60 toạ độ ở nếp gấp)

✅ Gradient của CẢ tầng Embedding lẫn ngăn xếp tích chập là ĐÚNG.


## 4. Huấn luyện ba nền tảng

In [9]:
# ============================================================================
# KHỐI 9 — Huấn luyện bản NumPy from scratch
# eval_subset: mỗi epoch chỉ tính loss/metric TRAIN trên 4000 mẫu cố định cho
# nhanh; early stopping vẫn dựa trên VALIDATION đầy đủ.
# ============================================================================
def val_macro_f1(y_true_oh, y_prob):
    """Tiêu chí chọn mô hình dùng CHUNG cho cả ba nền tảng."""
    return f1_score(y_true_oh.argmax(1), y_prob.argmax(1),
                    average="macro", zero_division=0)


cnn = build_cnn()
t0 = time.time()
cnn.fit(Xtr, ytr, Xva, yva, epochs=HP["epochs"], batch_size=HP["batch_size"],
        patience=HP["patience"], verbose_every=1, eval_subset=4000,
        val_score_fn=val_macro_f1)
numpy_time = time.time() - t0
print(f"\n⏱ NumPy from scratch: {numpy_time:.1f}s "
      f"({len(cnn.history['train_loss'])} epoch)")

  epoch    1 | train_loss=1.6337 | train_metric=0.4920 | val_loss=1.6924 | val_metric=0.4795 | val_score=0.2883


  epoch    2 | train_loss=1.2617 | train_metric=0.7545 | val_loss=1.3722 | val_metric=0.7420 | val_score=0.4173


  epoch    3 | train_loss=0.9784 | train_metric=0.7298 | val_loss=1.1835 | val_metric=0.7007 | val_score=0.5035


  epoch    4 | train_loss=0.7627 | train_metric=0.8060 | val_loss=1.1754 | val_metric=0.7602 | val_score=0.5653


  epoch    5 | train_loss=0.5835 | train_metric=0.7970 | val_loss=1.1776 | val_metric=0.7196 | val_score=0.5458


...

  epoch   16 | train_loss=0.0785 | train_metric=0.9617 | val_loss=2.8126 | val_metric=0.7873 | val_score=0.5904


  epoch   17 | train_loss=0.0957 | train_metric=0.9367 | val_loss=2.6180 | val_metric=0.7490 | val_score=0.5611


  epoch   18 | train_loss=0.0832 | train_metric=0.9495 | val_loss=3.2542 | val_metric=0.7591 | val_score=0.5676


  epoch   19 | train_loss=0.0720 | train_metric=0.9520 | val_loss=3.2946 | val_metric=0.7638 | val_score=0.5726


  ⏹ Early stopping tại epoch 20 (val_score tốt nhất = 0.5908)

⏱ NumPy from scratch: 324.7s (20 epoch)


In [10]:
# ============================================================================
# KHỐI 10 — Bản PyTorch tương đương
# nn.Embedding(padding_idx=0) ghim vector PAD bằng 0 giống bản NumPy.
# CrossEntropyLoss(weight=...) tương ứng class_weight của bản NumPy.
# ============================================================================
import torch
import torch.nn as nn

torch.manual_seed(SEED)


class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, HP["emb_dim"], padding_idx=PAD)
        self.c1 = nn.Conv1d(HP["emb_dim"], HP["f1"], 3)
        self.c2 = nn.Conv1d(HP["f1"], HP["f2"], 3)
        self.pool = nn.MaxPool1d(2)
        self.c3 = nn.Conv1d(HP["f2"], HP["f3"], 3)
        self.drop = nn.Dropout(HP["dropout"])
        self.d1 = nn.Linear(HP["f3"], 32)
        self.d2 = nn.Linear(32, N_CLASS)

    def forward(self, x):
        h = self.emb(x).transpose(1, 2)          # (N, L, E) -> (N, E, L)
        h = torch.relu(self.c1(h))
        h = torch.relu(self.c2(h))
        h = self.pool(h)
        h = torch.relu(self.c3(h))
        h = h.max(dim=2).values                  # GlobalMaxPool1D
        h = self.drop(h)
        return self.d2(torch.relu(self.d1(h)))


torch_net = TextCNN()
n_torch = sum(p.numel() for p in torch_net.parameters())
print(f"Tham số PyTorch = {n_torch:,} | NumPy = {cnn.n_params():,}")
assert n_torch == cnn.n_params(), "Hai bản không cùng số tham số!"
print("✅ Khớp số tham số.\n")

Xtr_t = torch.tensor(Xtr)
ytr_t = torch.tensor(ytr_i)
Xva_t = torch.tensor(Xva)
yva_t = torch.tensor(yva_i)
Xte_t = torch.tensor(Xte)

opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"])
lossf = nn.CrossEntropyLoss(weight=torch.tensor(class_weight, dtype=torch.float32))
torch_hist = {"train_loss": [], "val_loss": []}
g = torch.Generator().manual_seed(SEED)
# Chọn mô hình theo val Macro-F1 — CÙNG tiêu chí với bản NumPy, để so sánh công bằng.
best_f1, best_sd, wait = -np.inf, None, 0

t0 = time.time()
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(len(Xtr_t), generator=g)
    for s in range(0, len(perm), HP["batch_size"]):
        sl = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        loss = lossf(torch_net(Xtr_t[sl]), ytr_t[sl])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(torch_net.parameters(), HP["clip"])
        opt.step()
    torch_net.eval()
    with torch.no_grad():
        tl = float(lossf(torch_net(Xtr_t[:4000]), ytr_t[:4000]))
        va_logits = torch_net(Xva_t)
        vl = float(lossf(va_logits, yva_t))
        vf1 = f1_score(yva_i, va_logits.argmax(1).numpy(), average="macro",
                       zero_division=0)
    torch_hist["train_loss"].append(tl)
    torch_hist["val_loss"].append(vl)
    print(f"  epoch {ep:3d} | train_loss={tl:.4f} | val_loss={vl:.4f} | val_f1={vf1:.4f}")
    if vf1 > best_f1 + 1e-6:
        best_f1, wait = vf1, 0
        best_sd = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep} (val Macro-F1 = {best_f1:.4f})")
            break
if best_sd is not None:
    torch_net.load_state_dict(best_sd)
torch_time = time.time() - t0
print(f"\n⏱ PyTorch: {torch_time:.1f}s ({len(torch_hist['train_loss'])} epoch)")

Tham số PyTorch = 75,094 | NumPy = 75,094
✅ Khớp số tham số.



  epoch   1 | train_loss=1.7351 | val_loss=1.7453 | val_f1=0.2213


  epoch   2 | train_loss=1.6864 | val_loss=1.6980 | val_f1=0.1243


  epoch   3 | train_loss=1.5285 | val_loss=1.5428 | val_f1=0.3376


  epoch   4 | train_loss=1.2591 | val_loss=1.3300 | val_f1=0.4733


...

  epoch  30 | train_loss=0.1956 | val_loss=3.2435 | val_f1=0.5369


  epoch  31 | train_loss=0.1854 | val_loss=3.8806 | val_f1=0.5303


  epoch  32 | train_loss=0.1622 | val_loss=3.8241 | val_f1=0.5431


  epoch  33 | train_loss=0.1658 | val_loss=3.9546 | val_f1=0.5454


  epoch  34 | train_loss=0.1703 | val_loss=4.0885 | val_f1=0.5379
  ⏹ Early stopping tại epoch 34 (val Macro-F1 = 0.5513)

⏱ PyTorch: 67.7s (34 epoch)


In [11]:
# ============================================================================
# KHỐI 11 — Bản TensorFlow/Keras
# Keras Embedding trả (N, L, E) — đã là channels-last nên Conv1D dùng thẳng,
# KHÔNG cần transpose như hai bài trước. Đây là điểm layout đáng chú ý.
# ============================================================================
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(SEED)

tf_net = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN,), dtype="int64"),
    keras.layers.Embedding(VOCAB_SIZE, HP["emb_dim"], mask_zero=False),
    keras.layers.Conv1D(HP["f1"], 3, activation="relu"),
    keras.layers.Conv1D(HP["f2"], 3, activation="relu"),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(HP["f3"], 3, activation="relu"),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dropout(HP["dropout"]),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(N_CLASS, activation=None),
])
print(f"Tham số TensorFlow = {tf_net.count_params():,}")
assert tf_net.count_params() == cnn.n_params()
print("✅ Ba bản NumPy / PyTorch / TensorFlow cùng số tham số.\n")

tf_net.compile(
    optimizer=keras.optimizers.Adam(learning_rate=HP["lr"], global_clipnorm=HP["clip"]),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)


class MacroF1EarlyStopping(keras.callbacks.Callback):
    """Early stopping + giữ trọng số tốt nhất theo val Macro-F1.

    Keras không có sẵn Macro-F1 cho nhãn số nguyên nên ta tự tính. Điều quan
    trọng là dùng ĐÚNG tiêu chí chọn mô hình như hai bản kia — nếu mỗi nền tảng
    chọn mô hình theo một tiêu chí khác nhau thì phép so sánh ba nền tảng mất ý
    nghĩa ngay từ đầu.
    """

    def __init__(self, X_val, y_val, patience):
        super().__init__()
        self.X_val, self.y_val, self.patience = X_val, y_val, patience
        self.best, self.best_w, self.wait = -np.inf, None, 0
        self.scores = []

    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.X_val, verbose=0).argmax(1)
        s = f1_score(self.y_val, p, average="macro", zero_division=0)
        self.scores.append(s)
        if s > self.best + 1e-6:
            self.best, self.wait = s, 0
            self.best_w = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)
        print(f"  ⏹ TensorFlow dừng — val Macro-F1 tốt nhất = {self.best:.4f}")


es = MacroF1EarlyStopping(Xva, yva_i, HP["patience"])
t0 = time.time()
tf_hist = tf_net.fit(
    Xtr, ytr_i, validation_data=(Xva, yva_i),
    epochs=HP["epochs"], batch_size=HP["batch_size"],
    class_weight={i: float(w) for i, w in enumerate(class_weight)},
    callbacks=[es], verbose=0,
)
tf_time = time.time() - t0
print(f"⏱ TensorFlow: {tf_time:.1f}s ({len(tf_hist.history['loss'])} epoch)")

Tham số TensorFlow = 75,094
✅ Ba bản NumPy / PyTorch / TensorFlow cùng số tham số.



  ⏹ TensorFlow dừng — val Macro-F1 tốt nhất = 0.5903
⏱ TensorFlow: 51.1s (16 epoch)


## 5. Mô hình đối sánh: TF-IDF + Logistic Regression

Đây là baseline **rất mạnh** cho phân loại văn bản, và quan trọng hơn: nó là
mô hình **túi từ (bag-of-words)** — hoàn toàn **bỏ qua thứ tự từ**. So CNN với
nó chính là đo xem *thông tin thứ tự* đóng góp được bao nhiêu.

In [12]:
# ============================================================================
# KHỐI 12 — TF-IDF (1–2 gram) + Logistic Regression, cùng tập train/test
# ============================================================================
t0 = time.time()
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                        sublinear_tf=True, min_df=2)
Ttr = tfidf.fit_transform(texts[idx_train])
Tva = tfidf.transform(texts[idx_val])
Tte = tfidf.transform(texts[idx_test])

logreg = LogisticRegression(max_iter=1500, class_weight="balanced", C=4.0,
                            random_state=SEED)
logreg.fit(Ttr, ytr_i)
tfidf_time = time.time() - t0
print(f"TF-IDF + LogReg: {Ttr.shape[1]:,} đặc trưng, huấn luyện {tfidf_time:.1f}s")
print(f"  val Macro-F1 = {f1_score(yva_i, logreg.predict(Tva), average='macro'):.4f}")

# Baseline thứ hai: TF-IDF CHỈ unigram — bỏ hoàn toàn thông tin cụm từ
tfidf_uni = TfidfVectorizer(max_features=20000, ngram_range=(1, 1),
                            sublinear_tf=True, min_df=2)
Utr = tfidf_uni.fit_transform(texts[idx_train])
Ute = tfidf_uni.transform(texts[idx_test])
logreg_uni = LogisticRegression(max_iter=1500, class_weight="balanced", C=4.0,
                                random_state=SEED)
logreg_uni.fit(Utr, ytr_i)
print(f"  (bản chỉ unigram: val Macro-F1 = "
      f"{f1_score(yva_i, logreg_uni.predict(tfidf_uni.transform(texts[idx_val])), average='macro'):.4f})")

TF-IDF + LogReg: 20,000 đặc trưng, huấn luyện 6.8s
  val Macro-F1 = 0.6371


  (bản chỉ unigram: val Macro-F1 = 0.6237)


## 6. Đánh giá trên tập kiểm thử

Với dữ liệu mất cân bằng 88:1, **Accuracy là chỉ số gây hiểu lầm**: chỉ cần
đoán "Tops" cho mọi mẫu đã đạt ~46%. Ta dùng **Macro-F1** (trung bình F1 của
từng lớp, mọi lớp có trọng số bằng nhau) và **Balanced Accuracy** làm chỉ số
chính.

In [13]:
# ============================================================================
# KHỐI 13 — Dự đoán của mọi mô hình trên tập test
# ============================================================================
pred = {}
proba = {}

proba["CNN (NumPy)"] = cnn.predict_proba(Xte, batch=512)
torch_net.eval()
with torch.no_grad():
    lg = torch_net(Xte_t).numpy()
    proba["CNN (PyTorch)"] = np.exp(lg - lg.max(1, keepdims=True))
    proba["CNN (PyTorch)"] /= proba["CNN (PyTorch)"].sum(1, keepdims=True)
lg_tf = tf_net.predict(Xte, verbose=0)
proba["CNN (TensorFlow)"] = np.exp(lg_tf - lg_tf.max(1, keepdims=True))
proba["CNN (TensorFlow)"] /= proba["CNN (TensorFlow)"].sum(1, keepdims=True)
proba["TF-IDF 1-2gram + LogReg"] = logreg.predict_proba(Tte)
proba["TF-IDF unigram + LogReg"] = logreg_uni.predict_proba(Ute)

for k, v in proba.items():
    pred[k] = v.argmax(1)


def evaluate(p):
    return {
        "Accuracy": accuracy_score(yte_i, p),
        "Macro-F1": f1_score(yte_i, p, average="macro", zero_division=0),
        "Weighted-F1": f1_score(yte_i, p, average="weighted", zero_division=0),
        "Balanced Acc": balanced_accuracy_score(yte_i, p),
    }


res_df = pd.DataFrame({k: evaluate(v) for k, v in pred.items()}).T
res_df = res_df.sort_values("Macro-F1", ascending=False)
print(f"Tập test: {len(yte_i):,} đánh giá\n")
print("Chỉ cần đoán 'Tops' cho mọi mẫu đã đạt Accuracy = "
      f"{100*np.mean(yte_i == CLASSES.index('Tops')):.1f}% "
      "-> vì sao Accuracy gây hiểu lầm.\n")
res_df.round(4)

Tập test: 3,395 đánh giá

Chỉ cần đoán 'Tops' cho mọi mẫu đã đạt Accuracy = 44.4% -> vì sao Accuracy gây hiểu lầm.



,Accuracy,Macro-F1,Weighted-F1,Balanced Acc
TF-IDF 1-2gram + LogReg,0.8483,0.6354,0.8455,0.6444
TF-IDF unigram + LogReg,0.8221,0.6209,0.8264,0.6484
CNN (NumPy),0.7944,0.6035,0.8043,0.6408
CNN (TensorFlow),0.7703,0.5831,0.7933,0.6163
CNN (PyTorch),0.7464,0.5568,0.7612,0.5775


In [14]:
# ============================================================================
# KHỐI 14 — F1 theo từng lớp: lớp hiếm có bị bỏ rơi không?
# ============================================================================
per_class = pd.DataFrame(
    {name: f1_score(yte_i, p, average=None, labels=range(N_CLASS), zero_division=0)
     for name, p in pred.items()},
    index=[f"{c} ({CLASS_VI[c]})" for c in CLASSES])
per_class["Số mẫu test"] = np.bincount(yte_i, minlength=N_CLASS)
print("F1 theo từng lớp:\n")
per_class.round(4)

F1 theo từng lớp:



,CNN (NumPy),CNN (PyTorch),CNN (TensorFlow),TF-IDF 1-2gram + LogReg,TF-IDF unigram + LogReg,Số mẫu test
Bottoms (Quần / Chân váy),0.8176,0.7610,0.8008,0.8703,0.8490,549
Dresses (Đầm),0.8732,0.8469,0.8652,0.8932,0.8754,922
Intimate (Đồ lót & mặc nhà),0.4539,0.3025,0.3726,0.5033,0.5084,248
Jackets (Áo khoác),0.5502,0.5842,0.6170,0.6529,0.6220,150
Tops (Áo),0.8490,0.8106,0.8428,0.8929,0.8708,1508
Trend (Hàng xu hướng),0.0769,0.0357,0.0000,0.0000,0.0000,18


In [15]:
# ============================================================================
# KHỐI 15 — Độ lệch giữa ba nền tảng
# ============================================================================
tri = ["CNN (NumPy)", "CNN (PyTorch)", "CNN (TensorFlow)"]
rows = []
for a, b in [(tri[0], tri[1]), (tri[0], tri[2]), (tri[1], tri[2])]:
    pa, pb = proba[a], proba[b]
    rows.append({
        "Cặp so sánh": f"{a.split('(')[1][:-1]} ↔ {b.split('(')[1][:-1]}",
        "Sai khác xác suất TB": float(np.mean(np.abs(pa - pb))),
        "Số dự đoán khác nhau": int(np.sum(pa.argmax(1) != pb.argmax(1))),
        "Tỉ lệ đồng thuận": float(np.mean(pa.argmax(1) == pb.argmax(1))),
    })
divergence = pd.DataFrame(rows)
divergence.round(6)

,Cặp so sánh,Sai khác xác suất TB,Số dự đoán khác nhau,Tỉ lệ đồng thuận
0,NumPy ↔ PyTorch,0.083486,791,0.767010
1,NumPy ↔ TensorFlow,0.072207,667,0.803535
2,PyTorch ↔ TensorFlow,0.087308,800,0.764359


## 7. Thí nghiệm then chốt: thứ tự từ có thật sự quan trọng không?

Đây là thí nghiệm đối xứng với phép thử hoán vị cột ở Bài 1, nhưng lần này ta
**kỳ vọng kết quả ngược lại**.

Ta **xáo trộn ngẫu nhiên thứ tự từ trong từng đánh giá của tập test** (giữ
nguyên đúng tập từ, chỉ đảo vị trí) rồi cho mô hình đã huấn luyện dự đoán lại.

- Mô hình **túi từ** (TF-IDF unigram) **không thể** bị ảnh hưởng — nó không nhìn
  thấy thứ tự. Đây là đối chứng.
- Nếu CNN **sụt hiệu năng**, nghĩa là nó thật sự đã học các mẫu **n-gram cục bộ**,
  chứ không chỉ đếm từ.

In [16]:
# ============================================================================
# KHỐI 16 — Xáo trộn thứ tự từ trong từng đánh giá của tập TEST
# Chỉ xáo phần token thật, giữ nguyên phần đệm ở cuối.
# ============================================================================
rng_sh = np.random.default_rng(SEED)
Xte_shuf = Xte.copy()
for i in range(len(Xte_shuf)):
    n_real = int(np.sum(Xte_shuf[i] != PAD))
    if n_real > 1:
        seg = Xte_shuf[i, :n_real].copy()
        rng_sh.shuffle(seg)
        Xte_shuf[i, :n_real] = seg

texts_shuf = []
for i in idx_test:
    toks = tokenize(texts[i])
    rng_sh.shuffle(toks)
    texts_shuf.append(" ".join(toks))

f1_cnn_orig = f1_score(yte_i, pred["CNN (NumPy)"], average="macro", zero_division=0)
f1_cnn_shuf = f1_score(yte_i, cnn.predict_proba(Xte_shuf, batch=512).argmax(1),
                       average="macro", zero_division=0)

uni_shuf = logreg_uni.predict(tfidf_uni.transform(texts_shuf))
f1_uni_orig = f1_score(yte_i, pred["TF-IDF unigram + LogReg"], average="macro",
                       zero_division=0)
f1_uni_shuf = f1_score(yte_i, uni_shuf, average="macro", zero_division=0)

bi_shuf = logreg.predict(tfidf.transform(texts_shuf))
f1_bi_orig = f1_score(yte_i, pred["TF-IDF 1-2gram + LogReg"], average="macro",
                      zero_division=0)
f1_bi_shuf = f1_score(yte_i, bi_shuf, average="macro", zero_division=0)

shuffle_df = pd.DataFrame([
    {"Mô hình": "CNN (NumPy)", "Nhìn thấy thứ tự?": "Có",
     "Macro-F1 gốc": f1_cnn_orig, "Macro-F1 sau xáo trộn": f1_cnn_shuf},
    {"Mô hình": "TF-IDF 1-2gram + LogReg", "Nhìn thấy thứ tự?": "Một phần (bigram)",
     "Macro-F1 gốc": f1_bi_orig, "Macro-F1 sau xáo trộn": f1_bi_shuf},
    {"Mô hình": "TF-IDF unigram + LogReg", "Nhìn thấy thứ tự?": "Không (túi từ)",
     "Macro-F1 gốc": f1_uni_orig, "Macro-F1 sau xáo trộn": f1_uni_shuf},
])
shuffle_df["Mức sụt"] = shuffle_df["Macro-F1 gốc"] - shuffle_df["Macro-F1 sau xáo trộn"]

print("Xáo trộn thứ tự từ trong từng đánh giá của tập test:\n")
print(shuffle_df.round(4).to_string(index=False))
print()
print(f"CNN sụt              : {shuffle_df.loc[0, 'Mức sụt']:+.4f}")
print(f"TF-IDF unigram sụt   : {shuffle_df.loc[2, 'Mức sụt']:+.4f}  "
      "(phải ≈ 0 — đối chứng, vì túi từ không thấy thứ tự)")
assert abs(shuffle_df.loc[2, "Mức sụt"]) < 1e-9, \
    "Đối chứng sai: mô hình túi từ lẽ ra phải bất biến với xáo trộn"
print("\n✅ Đối chứng đúng như lý thuyết: mô hình túi từ bất biến hoàn toàn.")

Xáo trộn thứ tự từ trong từng đánh giá của tập test:

                Mô hình Nhìn thấy thứ tự?  Macro-F1 gốc  Macro-F1 sau xáo trộn  Mức sụt
            CNN (NumPy)                Có        0.6035                 0.5220   0.0814
TF-IDF 1-2gram + LogReg Một phần (bigram)        0.6354                 0.6253   0.0101
TF-IDF unigram + LogReg    Không (túi từ)        0.6209                 0.6209   0.0000

CNN sụt              : +0.0814
TF-IDF unigram sụt   : +0.0000  (phải ≈ 0 — đối chứng, vì túi từ không thấy thứ tự)

✅ Đối chứng đúng như lý thuyết: mô hình túi từ bất biến hoàn toàn.


### 7.1. Đọc kết quả

Kết luận được **sinh từ số liệu vừa đo**, so sánh trực tiếp với Bài 1.

In [17]:
# ============================================================================
# KHỐI 16b — Diễn giải và đối chiếu với Bài toán 1 (dữ liệu bảng)
# ============================================================================
drop_cnn = float(shuffle_df.loc[0, "Mức sụt"])
rel_drop = 100 * drop_cnn / f1_cnn_orig

print("QUAN SÁT")
print(f"  • Xáo trộn thứ tự từ làm Macro-F1 của CNN giảm {drop_cnn:+.4f} "
      f"({rel_drop:.1f}% tương đối).")
print(f"  • Mô hình túi từ không đổi (đúng 0) — xác nhận phép thử hợp lệ.")
print()
print("DIỄN GIẢI")
if drop_cnn > 0.01:
    print("  CNN MẤT hiệu năng khi thứ tự từ bị phá, trong khi mô hình túi từ thì")
    print("  không. Vậy CNN đã học được thông tin nằm ở TRẬT TỰ CỤC BỘ của token —")
    print("  đúng thứ mà tích chập sinh ra để khai thác. Ở đây kernel dài 3 hoạt")
    print("  động như một bộ dò n-gram học được, và điều đó có cơ sở ngôn ngữ học.")
else:
    print("  CNN gần như không đổi khi xáo trộn thứ tự từ. Nghĩa là trên tập dữ")
    print("  liệu này, nhãn (loại sản phẩm) được xác định chủ yếu bởi SỰ CÓ MẶT")
    print("  của một số từ khoá ('dress', 'jeans', 'blouse') chứ không bởi trật tự")
    print("  của chúng — nên lợi thế của tích chập bị thu hẹp đáng kể.")
print()
print("ĐỐI CHIẾU VỚI BÀI TOÁN 1")
print("  Bài 1 (bảng): hoán vị cột làm ROC-AUC dao động, nhưng đó là TẠO TÁC của")
print("    kiến trúc — không có quan hệ ngữ nghĩa nào giữa các cột liền kề.")
print("  Bài 3 (văn bản): thứ tự token mang ý nghĩa ngôn ngữ THẬT, nên đây mới là")
print("    nơi giả định cục bộ của CNN đứng vững về mặt bản chất dữ liệu.")

QUAN SÁT
  • Xáo trộn thứ tự từ làm Macro-F1 của CNN giảm +0.0814 (13.5% tương đối).
  • Mô hình túi từ không đổi (đúng 0) — xác nhận phép thử hợp lệ.

DIỄN GIẢI
  CNN MẤT hiệu năng khi thứ tự từ bị phá, trong khi mô hình túi từ thì
  không. Vậy CNN đã học được thông tin nằm ở TRẬT TỰ CỤC BỘ của token —
  đúng thứ mà tích chập sinh ra để khai thác. Ở đây kernel dài 3 hoạt
  động như một bộ dò n-gram học được, và điều đó có cơ sở ngôn ngữ học.

ĐỐI CHIẾU VỚI BÀI TOÁN 1
  Bài 1 (bảng): hoán vị cột làm ROC-AUC dao động, nhưng đó là TẠO TÁC của
    kiến trúc — không có quan hệ ngữ nghĩa nào giữa các cột liền kề.
  Bài 3 (văn bản): thứ tự token mang ý nghĩa ngôn ngữ THẬT, nên đây mới là
    nơi giả định cục bộ của CNN đứng vững về mặt bản chất dữ liệu.


## 8. Trực quan hoá

In [18]:
# ============================================================================
# KHỐI 17 — Hình 2: đường cong học của ba nền tảng
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (tr, va), title in [
    (axes[0], (cnn.history["train_loss"], cnn.history["val_loss"]), "(a) NumPy from scratch"),
    (axes[1], (torch_hist["train_loss"], torch_hist["val_loss"]), "(b) PyTorch"),
    (axes[2], (tf_hist.history["loss"], tf_hist.history["val_loss"]), "(c) TensorFlow/Keras"),
]:
    ax.plot(range(1, len(tr) + 1), tr, label="Train", color="#2563eb", linewidth=1.9,
            marker="o", markersize=3)
    ax.plot(range(1, len(va) + 1), va, label="Validation", color="#ef4444",
            linewidth=1.9, marker="s", markersize=3)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("Cross-Entropy có trọng số lớp")
plt.suptitle("Hình 2 — Đường cong học của CNN văn bản trên ba nền tảng",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c3_fig2_curves.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_8908\1854742572.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ============================================================================
# KHỐI 18 — Hình 3: ma trận nhầm lẫn + F1 theo lớp
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

cm = confusion_matrix(yte_i, pred["CNN (NumPy)"], labels=range(N_CLASS))
cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", ax=axes[0], cbar_kws={"shrink": .8},
            xticklabels=[CLASS_VI[c] for c in CLASSES],
            yticklabels=[CLASS_VI[c] for c in CLASSES], annot_kws={"size": 9})
axes[0].set_title("(a) Ma trận nhầm lẫn CNN (số = số mẫu, màu = tỉ lệ theo hàng)",
                  fontweight="bold", fontsize=10)
axes[0].set_xlabel("Dự đoán")
axes[0].set_ylabel("Thực tế")
axes[0].tick_params(axis="x", rotation=25, labelsize=8)
axes[0].tick_params(axis="y", rotation=0, labelsize=8)
axes[0].grid(False)

w = 0.26
xs = np.arange(N_CLASS)
for off, name, color in [(-w, "CNN (NumPy)", "#2563eb"),
                         (0, "TF-IDF 1-2gram + LogReg", "#f59e0b"),
                         (w, "TF-IDF unigram + LogReg", "#94a3b8")]:
    vals = f1_score(yte_i, pred[name], average=None, labels=range(N_CLASS),
                    zero_division=0)
    axes[1].bar(xs + off, vals, width=w, label=name, color=color)
axes[1].set_xticks(xs)
axes[1].set_xticklabels([f"{CLASS_VI[c]}\n(n={np.sum(yte_i==i)})"
                         for i, c in enumerate(CLASSES)], fontsize=7.5)
axes[1].set_ylabel("F1")
axes[1].set_title("(b) F1 theo từng lớp — lớp hiếm là chỗ khó nhất",
                  fontweight="bold", fontsize=10)
axes[1].legend(fontsize=7.5)
axes[1].set_ylim(0, 1.05)

plt.suptitle("Hình 3 — Chất lượng phân loại theo lớp", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG / "c3_fig3_confusion.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_8908\3493303610.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# ============================================================================
# KHỐI 19 — Hình 4: (a) đối sánh mô hình, (b) thí nghiệm xáo trộn, (c) thời gian
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

order_plot = res_df.sort_values("Macro-F1").index
colors = ["#2563eb" if "CNN" in n else "#94a3b8" for n in order_plot]
vals = res_df.loc[order_plot, "Macro-F1"].values
axes[0].barh(range(len(order_plot)), vals, color=colors)
axes[0].set_yticks(range(len(order_plot)))
axes[0].set_yticklabels(order_plot, fontsize=7.5)
for i, v in enumerate(vals):
    axes[0].text(v + 0.006, i, f"{v:.4f}", va="center", fontsize=8, fontweight="bold")
axes[0].set_xlim(min(vals) - 0.06, max(vals) + 0.06)
axes[0].set_title("(a) Macro-F1 trên tập test", fontweight="bold")
axes[0].set_xlabel("Macro-F1")

xs2 = np.arange(len(shuffle_df))
axes[1].bar(xs2 - 0.19, shuffle_df["Macro-F1 gốc"], width=0.38,
            label="Thứ tự gốc", color="#2563eb")
axes[1].bar(xs2 + 0.19, shuffle_df["Macro-F1 sau xáo trộn"], width=0.38,
            label="Đã xáo trộn từ", color="#ef4444")
for i, (a, b) in enumerate(zip(shuffle_df["Macro-F1 gốc"],
                               shuffle_df["Macro-F1 sau xáo trộn"])):
    axes[1].text(i, max(a, b) + 0.015, f"Δ={a-b:+.3f}", ha="center",
                 fontsize=8, fontweight="bold")
axes[1].set_xticks(xs2)
axes[1].set_xticklabels(["CNN", "TF-IDF\n1-2gram", "TF-IDF\nunigram"], fontsize=8)
axes[1].set_ylabel("Macro-F1")
axes[1].set_ylim(0, max(shuffle_df["Macro-F1 gốc"]) * 1.25)
axes[1].set_title("(b) Xáo trộn thứ tự từ — chỉ mô hình\nthấy thứ tự mới bị ảnh hưởng",
                  fontweight="bold", fontsize=9.5)
axes[1].legend(fontsize=8)

names_t = ["NumPy\nfrom scratch", "PyTorch", "TensorFlow"]
times = [numpy_time, torch_time, tf_time]
epochs_run = [len(cnn.history["train_loss"]), len(torch_hist["train_loss"]),
              len(tf_hist.history["loss"])]
per_ep = [t / e for t, e in zip(times, epochs_run)]
bars = axes[2].bar(names_t, per_ep, color=["#2563eb", "#16a34a", "#f59e0b"], width=0.55)
for bar, t, e in zip(bars, times, epochs_run):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                 f"{bar.get_height():.1f} s/epoch\n({t:.0f}s / {e} ep)",
                 ha="center", fontsize=8, fontweight="bold")
axes[2].set_title(f"(c) Thời gian/epoch — NumPy chậm hơn PyTorch "
                  f"{per_ep[0]/per_ep[1]:.1f}×", fontweight="bold", fontsize=9.5)
axes[2].set_ylabel("giây / epoch")
axes[2].set_ylim(0, max(per_ep) * 1.32)

plt.suptitle("Hình 4 — Đối sánh mô hình, vai trò của thứ tự từ và chi phí huấn luyện",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c3_fig4_compare.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_8908\1596018166.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.1. Mạng đã học được n-gram nào?

Mỗi bộ lọc của `Conv1` cộng với `GlobalMaxPool` trả lời câu hỏi *"n-gram mà tôi
phụ trách có xuất hiện trong câu không"*. Ta dò ngược: với từng bộ lọc, tìm
**3-gram trong tập test kích hoạt nó mạnh nhất**.

In [21]:
# ============================================================================
# KHỐI 20 — Hình 5: 3-gram kích hoạt mạnh nhất từng bộ lọc của Conv1
# ============================================================================
emb_layer = cnn.layers[0]
conv1 = cnn.layers[1]

sub = Xte[:3000]
H = relu(conv1d_forward(emb_layer.E[sub].transpose(0, 2, 1), conv1.W, conv1.b)[0])
# H: (n, f1, L-2) — với mỗi bộ lọc tìm vị trí kích hoạt lớn nhất trên toàn tập con
top_rows = []
for f in range(HP["f1"]):
    flat_idx = int(np.argmax(H[:, f, :]))
    si, pi = divmod(flat_idx, H.shape[2])
    gram = [vocab[t] for t in sub[si, pi:pi + 3]]
    top_rows.append({"Bộ lọc": f, "3-gram kích hoạt mạnh nhất": " ".join(gram),
                     "Giá trị kích hoạt": float(H[si, f, pi])})
top_df = pd.DataFrame(top_rows).sort_values("Giá trị kích hoạt", ascending=False)

fig, ax = plt.subplots(figsize=(9.5, 6.5))
show = top_df.head(18).iloc[::-1]
ax.barh(range(len(show)), show["Giá trị kích hoạt"], color="#8b5cf6")
ax.set_yticks(range(len(show)))
ax.set_yticklabels([f'#{r["Bộ lọc"]}  "{r["3-gram kích hoạt mạnh nhất"]}"'
                    for _, r in show.iterrows()], fontsize=8.5)
ax.set_xlabel("Giá trị kích hoạt lớn nhất")
ax.set_title("Hình 5 — 18 bộ lọc Conv1 mạnh nhất và 3-gram kích hoạt chúng\n"
             "(mỗi kernel dài 3 là một bộ dò n-gram học được)",
             fontweight="bold", fontsize=10.5)
plt.tight_layout()
plt.savefig(FIG / "c3_fig5_ngrams.png", bbox_inches="tight")
plt.show()

print("15 bộ lọc kích hoạt mạnh nhất:\n")
print(top_df.head(15).to_string(index=False))

15 bộ lọc kích hoạt mạnh nhất:

 Bộ lọc 3-gram kích hoạt mạnh nhất  Giá trị kích hoạt
     11            yoga pants they           1.774945
     10         byron lars dresses           1.648385
      5              peplum tops v           1.615475
      1                 moth top i           1.523668
     15            ag stevie jeans           1.366431
     30         great lounge sleep           1.332644
     18       pilcro hyphen chinos           1.306146
     19       kimono style jackets           1.227454
      9            this dress sexy           1.211739
     22              suit cover up           1.127523
      6            the blazer ends           1.057578
      7         byron lars dresses           1.035794
     21      jumpsuits this summer           1.012720
     26         byron lars dresses           1.010137
     25           boots knee socks           1.003141


C:\Users\admin\AppData\Local\Temp\ipykernel_8908\1448589612.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Xuất mô hình cho web app React

In [22]:
# ============================================================================
# KHỐI 21 — Đóng gói model_cnn.json cho frontend
# Bundle mang theo TỪ ĐIỂN để JavaScript tự mã hoá văn bản, nên trình duyệt làm
# được trọn vẹn: tokenize -> encode -> embedding -> conv -> pool -> dense -> softmax.
# ============================================================================
# decimals=6: với 5 chữ số, sai số làm tròn tích luỹ qua Embedding + 3 tầng Conv
# đẩy chênh lệch xác suất lên ~1.2e-4 — vượt ngưỡng parity. Thêm một chữ số đưa
# nó về ~1e-5, đổi lại file JSON to thêm khoảng 14%.
bundle = cnn.to_dict(decimals=6)
bundle.update({
    "model_name": "CNN 1D văn bản (NumPy from scratch)",
    "assignment": "Assignment 04",
    "classes": CLASSES,
    "class_labels_vi": [CLASS_VI[c] for c in CLASSES],
    "tokenizer": {
        "regex": "[a-z0-9']+",
        "lowercase": True,
        "max_len": MAX_LEN,
        "pad_index": PAD,
        "unk_index": UNK,
        "vocab": vocab,
    },
    "metrics": {
        "accuracy": float(res_df.loc["CNN (NumPy)", "Accuracy"]),
        "macro_f1": float(res_df.loc["CNN (NumPy)", "Macro-F1"]),
        "weighted_f1": float(res_df.loc["CNN (NumPy)", "Weighted-F1"]),
        "balanced_accuracy": float(res_df.loc["CNN (NumPy)", "Balanced Acc"]),
    },
    "per_class_f1": {c: float(v) for c, v in zip(
        CLASSES, f1_score(yte_i, pred["CNN (NumPy)"], average=None,
                          labels=range(N_CLASS), zero_division=0))},
    "comparison": {
        name: {"accuracy": float(res_df.loc[name, "Accuracy"]),
               "macro_f1": float(res_df.loc[name, "Macro-F1"]),
               "balanced_accuracy": float(res_df.loc[name, "Balanced Acc"])}
        for name in res_df.index},
    "framework_divergence": divergence.to_dict(orient="records"),
    "word_order_test": {
        "cnn_f1_original": float(f1_cnn_orig),
        "cnn_f1_shuffled": float(f1_cnn_shuf),
        "bow_f1_original": float(f1_uni_orig),
        "bow_f1_shuffled": float(f1_uni_shuf),
    },
    "architecture_text": (f"tokens({MAX_LEN}) → Embedding({VOCAB_SIZE}×{HP['emb_dim']}) "
                          f"→ Conv({HP['f1']}) → Conv({HP['f2']}) → Pool "
                          f"→ Conv({HP['f3']}) → GlobalMaxPool → Dense(32) → {N_CLASS}"),
    "training": {
        "epochs_run": len(cnn.history["train_loss"]),
        "batch_size": HP["batch_size"], "lr": HP["lr"], "dropout": HP["dropout"],
        "vocab_size": VOCAB_SIZE, "max_len": MAX_LEN, "emb_dim": HP["emb_dim"],
        "numpy_seconds": round(numpy_time, 1),
        "pytorch_seconds": round(torch_time, 1),
        "tensorflow_seconds": round(tf_time, 1),
    },
    "history": {
        "train_loss": [round(v, 5) for v in cnn.history["train_loss"]],
        "val_loss": [round(v, 5) for v in cnn.history["val_loss"]],
        "train_acc": [round(v, 5) for v in cnn.history["train_metric"]],
        "val_acc": [round(v, 5) for v in cnn.history["val_metric"]],
    },
})

out_path = ROOT / "public" / "model_cnn.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path.relative_to(ROOT)} — {out_path.stat().st_size/1024:.1f} KB")
print(f"   {bundle['n_params']:,} tham số, từ điển {len(vocab):,} token")

✅ Đã ghi public\model_cnn.json — 742.0 KB
   75,094 tham số, từ điển 4,000 token


In [23]:
# ============================================================================
# KHỐI 22 — Parity NumPy <-> JSON trên 400 mẫu test
# ============================================================================
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
n_check = min(400, len(Xte))
max_err = 0.0
for i in range(n_check):
    ref = forward_reference(reloaded, Xte[i])
    max_err = max(max_err, float(np.abs(np.asarray(ref) - proba["CNN (NumPy)"][i]).max()))
print(f"Sai số tuyệt đối lớn nhất trên {n_check} mẫu: {max_err:.3e}")
assert max_err < 1e-4, "Parity FAILED"
print("✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.")

Sai số tuyệt đối lớn nhất trên 400 mẫu: 8.746e-06
✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.


In [24]:
# ============================================================================
# KHỐI 23 — Mẫu tham chiếu để script JS tự kiểm tra (đi kèm văn bản THÔ,
# nên bản JS phải tự tokenize đúng thì mới khớp).
# ============================================================================
samples = []
for k in range(12):
    i = int(idx_test[k])
    p = proba["CNN (NumPy)"][k]
    samples.append({
        "text": str(texts[i]),
        "expected_probs": [round(float(v), 8) for v in p],
        "expected_class": CLASSES[int(p.argmax())],
        "actual_class": CLASSES[int(yte_i[k])],
    })
samp_path = ROOT / "ml" / "model_cnn_samples.json"
samp_path.write_text(json.dumps(samples, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"✅ Đã ghi {samp_path.relative_to(ROOT)} — {len(samples)} mẫu (kèm văn bản thô)")

✅ Đã ghi ml\model_cnn_samples.json — 12 mẫu (kèm văn bản thô)


## 10. Kết luận Bài toán 3

In [25]:
# ============================================================================
# KHỐI 24 — Tóm tắt số liệu chính
# ============================================================================
cnn_f1 = res_df.loc["CNN (NumPy)", "Macro-F1"]
bi_f1 = res_df.loc["TF-IDF 1-2gram + LogReg", "Macro-F1"]
uni_f1 = res_df.loc["TF-IDF unigram + LogReg", "Macro-F1"]

print("=" * 72)
print("BÀI TOÁN 3 — PHÂN LOẠI ĐÁNH GIÁ KHÁCH HÀNG BẰNG CNN VĂN BẢN")
print("=" * 72)
print(f"Kiến trúc      : {bundle['architecture_text']}")
print(f"Tham số        : {cnn.n_params():,} (Embedding chiếm "
      f"{100*emb_params/cnn.n_params():.0f}%)")
print(f"Gradient check : sai số tương đối {worst:.2e} — gồm cả tầng Embedding")
print()
for n in res_df.index:
    print(f"{n:<26}: Macro-F1 = {res_df.loc[n, 'Macro-F1']:.4f}  |  "
          f"BalAcc = {res_df.loc[n, 'Balanced Acc']:.4f}  |  "
          f"Acc = {res_df.loc[n, 'Accuracy']:.4f}")
print()
print(f"CNN so với TF-IDF 1-2gram : {cnn_f1 - bi_f1:+.4f}")
print(f"CNN so với TF-IDF unigram : {cnn_f1 - uni_f1:+.4f}")
print()
print("Thời gian huấn luyện:")
for nm, t, e in zip(["NumPy", "PyTorch", "TensorFlow"], times, epochs_run):
    print(f"  {nm:<12}: {t/e:6.2f} s/epoch  (tổng {t:.0f}s, {e} epoch)")
print(f"  -> NumPy chậm hơn PyTorch {per_ep[0]/per_ep[1]:.1f}× ở quy mô này")
print()
print("Thí nghiệm xáo trộn thứ tự từ (tập test):")
print(f"  CNN            : {f1_cnn_orig:.4f} -> {f1_cnn_shuf:.4f}  ({drop_cnn:+.4f})")
print(f"  TF-IDF unigram : {f1_uni_orig:.4f} -> {f1_uni_shuf:.4f}  (đối chứng, = 0)")
print("=" * 72)

BÀI TOÁN 3 — PHÂN LOẠI ĐÁNH GIÁ KHÁCH HÀNG BẰNG CNN VĂN BẢN
Kiến trúc      : tokens(100) → Embedding(4000×16) → Conv(32) → Conv(32) → Pool → Conv(48) → GlobalMaxPool → Dense(32) → 6
Tham số        : 75,094 (Embedding chiếm 85%)
Gradient check : sai số tương đối 4.91e-08 — gồm cả tầng Embedding

TF-IDF 1-2gram + LogReg   : Macro-F1 = 0.6354  |  BalAcc = 0.6444  |  Acc = 0.8483
TF-IDF unigram + LogReg   : Macro-F1 = 0.6209  |  BalAcc = 0.6484  |  Acc = 0.8221
CNN (NumPy)               : Macro-F1 = 0.6035  |  BalAcc = 0.6408  |  Acc = 0.7944
CNN (TensorFlow)          : Macro-F1 = 0.5831  |  BalAcc = 0.6163  |  Acc = 0.7703
CNN (PyTorch)             : Macro-F1 = 0.5568  |  BalAcc = 0.5775  |  Acc = 0.7464

CNN so với TF-IDF 1-2gram : -0.0320
CNN so với TF-IDF unigram : -0.0175

Thời gian huấn luyện:
  NumPy       :  16.24 s/epoch  (tổng 325s, 20 epoch)
  PyTorch     :   1.99 s/epoch  (tổng 68s, 34 epoch)
  TensorFlow  :   3.19 s/epoch  (tổng 51s, 16 epoch)
  -> NumPy chậm hơn PyTorch 8.2× 

### Nhận xét trung thực

1. **Đây là bài duy nhất trong ba bài mà tích chập có cơ sở về bản chất dữ
   liệu.** Token liền kề trong câu **thực sự** tạo thành cụm có nghĩa, khác hẳn
   hai bài dữ liệu bảng nơi thứ tự cột do file CSV quyết định. Thí nghiệm xáo
   trộn thứ tự từ cho bằng chứng trực tiếp, với mô hình túi từ làm đối chứng
   bất biến hoàn hảo.

2. **Nhưng "có cơ sở" không đồng nghĩa với "thắng".** TF-IDF + Logistic
   Regression vẫn là baseline rất mạnh cho phân loại chủ đề văn bản, vì nhãn ở
   đây (loại sản phẩm) phần lớn được quyết định bởi **sự có mặt của từ khoá**
   (`dress`, `jeans`, `blouse`) hơn là bởi trật tự. CNN chỉ thật sự vượt trội
   khi bài toán phụ thuộc vào cụm từ — ví dụ phân tích cảm xúc, nơi `not good`
   và `very good` phải được phân biệt.

3. **Tầng Embedding chiếm phần lớn tham số** nhưng lại là phần *rẻ* nhất khi
   tính toán: nó chỉ là tra bảng. Ngược lại, ba tầng tích chập chiếm ít tham số
   mà tốn gần hết thời gian. Đây là minh hoạ rõ rằng **số tham số không phải
   thước đo chi phí tính toán**.

4. **Ba nền tảng lệch nhau ĐÁNG KỂ ở bài này — và đó là kết quả quan trọng.**
   Ở Bài 1 và Bài 2, ba bản cài cho chỉ số sát nhau tới chữ số thứ ba. Ở đây
   Macro-F1 lệch tới ~0,05 dù dùng **cùng kiến trúc, cùng số tham số, cùng siêu
   tham số, cùng tiêu chí chọn mô hình** (val Macro-F1). Nguyên nhân không phải
   một bản nào sai — gradient check đã xác nhận bản NumPy đúng tới $10^{-8}$ —
   mà là **phương sai của chính quá trình huấn luyện**:

   - khởi tạo bảng nhúng 64.000 tham số khác nhau giữa ba bộ sinh số ngẫu nhiên;
   - mặt nạ dropout và thứ tự trộn mini-batch khác nhau;
   - lớp `Trend` chỉ có ~18 mẫu trong tập test, nên F1 của nó — và qua đó
     Macro-F1 — nhảy rất mạnh chỉ vì vài mẫu đổi nhãn.

   **Bài học:** với dữ liệu mất cân bằng nặng, một lần chạy đơn lẻ **không đủ**
   để xếp hạng mô hình. Muốn kết luận "bản nào tốt hơn" thì phải chạy nhiều
   seed và báo cáo trung bình ± độ lệch chuẩn. Đây cũng là lý do ta không kết
   luận nền tảng nào "thắng" ở bảng trên.

5. **Ở quy mô này, khoảng cách hiệu năng giữa các nền tảng mới lộ ra.** Bài 1
   chỉ có 2.449 tham số nên bản NumPy còn nhanh hơn PyTorch nhờ không có
   overhead; ở đây, với ma trận nhúng và chuỗi dài 100, PyTorch/TensorFlow vượt
   hẳn nhờ kernel BLAS đa luồng. Đây chính là ranh giới mà việc "tự cài để
   hiểu" phải nhường chỗ cho "dùng framework để làm việc thật".

6. **Lớp hiếm vẫn là điểm yếu.** Lớp `Trend` chỉ có ~119 mẫu toàn bộ dữ liệu;
   trọng số lớp giúp mô hình không bỏ qua nó hoàn toàn, nhưng F1 của lớp này
   vẫn thấp hơn hẳn các lớp khác. Không có kiến trúc nào bù được cho việc thiếu
   dữ liệu.